In [6]:
include("src/polyatree.jl")
include("src/partitions.jl")

using Distributions
using Plots
using Serialization
using DataFrames
using LaTeXStrings
using Plots.PlotMeasures
using Printf

In [7]:
data1 = rand(Uniform(0, 0.5), 2000)
data2 = [rand(Uniform(0, 0.25), 1000); rand(Uniform(0.125, 0.25), 3000); rand(Uniform(0.5, 1.0), 2000)];
data3 = rand(truncated(Normal(0.5, 0.1), 0.0, 1.0), 1000);

datasets = [data1, data2, data3];

In [8]:
function credible_band_pt(
        xgrid::AbstractVector{<:Real}, 
        pt::Union{GFPT1, PolyaTree},
        K::Int=2000, level::Float64=0.95)
    @assert 0 < level < 1
    D = length(xgrid)

    densities = Matrix{Float64}(undef, D, K)
    logscores = Vector{Float64}(undef, K)

    for k in 1:K
        d, l = sample_pt_density(xgrid, pt, true)
        densities[:, k] = d
        logscores[k] = l
    end
    # keep top ceil(level*K) draws by log posterior score
    idx = sortperm(logscores, rev=true)
    keep = idx[1:cld(round(Int, level*K), 1)]
    dens_keep = densities[:, keep]
    lower = mapslices(minimum, dens_keep; dims=2)[:]
    upper = mapslices(maximum, dens_keep; dims=2)[:]
    meta = (; kept=keep, method=:HPD)
    return lower, upper, densities
end

credible_band_pt (generic function with 3 methods)

In [9]:
alpha0 = 2.0
xgrid = collect(LinRange(1e-4, 1.0 - 1e-6, 50))
gfpt1 = GFPT1(Poisson(2.0), 10, alpha0)
gfpt1 = update(data1, gfpt1)
lower, upper, densities = credible_band_pt(xgrid, gfpt1);

In [16]:
xgrid = collect(LinRange(1e-4, 1.0 - 1e-6, 1000))
alpha_0s = [0.05, 0.1, 2.0, 10.0]

data = data3
plots = []
for a in alpha_0s
    p = plot()

    gfpt1 = GFPT1(Poisson(5.0), 10, a)
    pt = deepcopy(gfpt1.pt)
    pt = update(data, pt)
    lower, upper, meta = credible_band_pt(xgrid, pt)
    mean = predictive_density(xgrid, pt)
    t = L"$\alpha_0 = $"*@sprintf("%.2f", a)
    plot!(xgrid, mean; ribbon=(mean .- lower, upper .- mean), label="PT", title=t)

    gfpt1 = GFPT1(Poisson(10.0), 20, a)
    gfpt1 = update(data, gfpt1)
    lower, upper, meta = credible_band_pt(xgrid, gfpt1)
    mean = predictive_density(xgrid, gfpt1)
    plot!(xgrid, mean; ribbon=(mean .- lower, upper .- mean), label="GFPT")


    plots = push!(plots, p)
end

In [18]:
plot(plots..., layout=(1, 4), size=(800, 250))
savefig("plots/example2_3.pdf")

"/Users/marioberaha/Documents/uni/papers/polyatree/code/plots/example2_3.pdf"

In [20]:
xgrid = collect(LinRange(1e-4, 1.0 - 1e-6, 1000))
alpha_0s = [0.05, 0.1, 2.0, 10.0]

plots = []
for a in alpha_0s
    p = plot()

    gfpt1 = GFPT1(Poisson(5.0), 10, a)
    pt = deepcopy(gfpt1.pt)
    lower, upper, meta = credible_band_pt(xgrid, pt)
    mean = predictive_density(xgrid, pt)
    t = L"$\alpha_0 = $"*@sprintf("%.2f", a)
    plot!(xgrid, mean; ribbon=(mean .- lower, upper .- mean), label="PT", title=t)

    gfpt1 = GFPT1(Poisson(5.0), 10, a)
    lower, upper, meta = credible_band_pt(xgrid, gfpt1)
    mean = predictive_density(xgrid, gfpt1)
    plot!(xgrid, mean; ribbon=(mean .- lower, upper .- mean), label="GFPT")

    plots = push!(plots, p)
end

plot(plots..., layout=(1, 4), size=(800, 250))
savefig("plots/example2_prior.pdf")

"/Users/marioberaha/Documents/uni/papers/polyatree/code/plots/example2_prior.pdf"